# Read from CSV File

In [0]:
df = (spark.read.option("header", "true").option("inferSchema", "true").csv("/Volumes/dev_project/bronze/source_system/cust_info.csv"))

df.display()

## Repeat with the rest of the files

In [0]:
df_prd = (spark.read.option("header", "true").option("inferSchema", "true").csv("/Volumes/dev_project/bronze/source_system/prd_info.csv"))

df_sales = (spark.read.option("header", "true").option("inferSchema", "true").csv("/Volumes/dev_project/bronze/source_system/sales_details.csv"))

df_cust = (spark.read.option("header", "true").option("inferSchema", "true").csv("/Volumes/dev_project/bronze/source_system/CUST_AZ12.csv"))

df_loc = (spark.read.option("header", "true").option("inferSchema", "true").csv("/Volumes/dev_project/bronze/source_system/LOC_A101.csv"))

df_px = (spark.read.option("header", "true").option("inferSchema", "true").csv("/Volumes/dev_project/bronze/source_system/PX_CAT_G1V2.csv"))

# Create Delta Table in Bronze Layer

In [0]:
df.write.mode("overwrite").saveAsTable("dev_project.bronze.crm_cust_info")

In [0]:
%sql
select
*
from
dev_project.bronze.crm_cust_info

## Repeat with the rest DataFrames

In [0]:
df_prd.write.mode("overwrite").saveAsTable("dev_project.bronze.crm_prd_info")
df_sales.write.mode("overwrite").saveAsTable("dev_project.bronze.crm_sales_details")
df_cust.write.mode("overwrite").saveAsTable("dev_project.bronze.erp_cust")
df_loc.write.mode("overwrite").saveAsTable("dev_project.bronze.erp_loc")
df_px.write.mode("overwrite").saveAsTable("dev_project.bronze.erp_px")


## Alternative way to process the files
Create configuration file (e.g. bronze_configuration) with a list of dicts. Each dict includes:
- "source" - system the file received from (crm / erp)
- "path" - location to the file
- "table" - name of the delta table where inforamtion will be uploaded

In [0]:
from bronze_config import INGESTION_CONFIG

for item in INGESTION_CONFIG:
    print(f"Ingesting {item["source"]} -> bronze.{item["table"]}")

    df = (
        spark.read
            .option("header", "true")
            .option("inferSchema", "true")
            .csv(item["path"])
    )
    (
    df.write
        .mode("overwrite")
        .saveAsTable(f"dev_project.bronze.{item["table"]}")
    )
    print(f"Success. Table f{item["table"]} is updates.")